# Document Summary RAG with AWS

## 📖 Overview

**Document Summary RAG** combines document-level summaries with chunk-level details for efficient two-stage retrieval.

### Pattern: Summary → Detail

```
Stage 1: Search summaries (fast, document-level)
         ↓
Stage 2: Retrieve detailed chunks from relevant documents
         ↓
Generate: Use both summary + details
```

### Key Benefits

- **Fast Filtering**: Summary search narrows to relevant documents
- **Rich Context**: Detail chunks provide specifics
- **Lower Cost**: Search fewer chunks
- **Better Precision**: Two-stage filtering

### Architecture

```mermaid
graph TB
    A[Documents] --> B[Generate Summaries]
    A --> C[Create Detail Chunks]
    
    B --> D[Summary Index<br/>Document-level]
    C --> E[Detail Index<br/>Chunk-level]
    
    F[Query] --> G[Stage 1:<br/>Search Summaries]
    G --> D
    D --> H[Top-K Documents]
    
    H --> I[Stage 2:<br/>Get Details]
    I --> E
    E --> J[Detailed Chunks]
    
    J --> K[Generate Answer]
    
    style A fill:#e1f5ff
    style D fill:#fff3e0
    style E fill:#e8f5e9
    style H fill:#f3e5f5
    style K fill:#b3e5fc
```

## 1️⃣ Setup

In [ ]:
import sys
import json
from typing import List, Dict
import time

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
AWS_REGION = 'us-west-2'
SUMMARY_INDEX = 'doc_summaries'
DETAIL_INDEX = 'doc_details'

EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
LLM_MODEL = 'us.anthropic.claude-sonnet-4-6'

# Two-stage parameters
SUMMARY_TOP_K = 3  # Stage 1: Top documents
DETAIL_TOP_K = 5   # Stage 2: Chunks per document

print(f"Configuration:")
print(f"  Summary retrieval: Top-{SUMMARY_TOP_K} docs")
print(f"  Detail retrieval: Top-{DETAIL_TOP_K} chunks/doc")

## 3️⃣ Initialize Services

In [ ]:
summary_index = OpenSearchManager(region=AWS_REGION, index_name=SUMMARY_INDEX)
detail_index = OpenSearchManager(region=AWS_REGION, index_name=DETAIL_INDEX)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

## 4️⃣ Create Document Summaries

In [ ]:
# Sample documents with full content
documents = [
    {
        'doc_id': 'aws_bedrock_guide',
        'title': 'AWS Bedrock Complete Guide',
        'content': '''AWS Bedrock is a fully managed service providing foundation model access. 
Claude models offer three tiers: Opus ($15/$75 per million tokens), Sonnet ($3/$15), and Haiku ($0.25/$1.25). 
Bedrock integrates with SageMaker and Lambda for complete ML workflows. 
Best for: production AI applications requiring managed infrastructure.'''
    },
    {
        'doc_id': 'opensearch_rag',
        'title': 'OpenSearch for RAG',
        'content': '''OpenSearch Serverless provides vector search without cluster management. 
Uses HNSW algorithm for fast similarity search. Costs ~$0.24/hour for compute. 
Auto-scales based on workload. Perfect for RAG applications. 
Integrates with Bedrock embeddings for semantic search.'''
    },
    {
        'doc_id': 'lambda_orchestration',
        'title': 'Lambda RAG Orchestration',
        'content': '''AWS Lambda orchestrates RAG pipelines serverlessly. 
Pricing: $0.20 per million requests + compute time. 
Handles: query processing, retrieval coordination, response formatting. 
Cold starts: 100-500ms, use provisioned concurrency for critical paths.'''
    }
]

# Generate summaries
print("Generating document summaries...\n")

summaries = []
for doc in documents:
    prompt = f"""Summarize this document in 1-2 sentences focusing on key topics:

Title: {doc['title']}
Content: {doc['content']}

Summary:"""
    
    summary = llm.generate(prompt, temperature=0.3)
    summaries.append({
        'doc_id': doc['doc_id'],
        'title': doc['title'],
        'summary': summary,
        'full_content': doc['content']
    })
    print(f"✓ {doc['title']}")
    print(f"  Summary: {summary[:100]}...\n")

print(f"✓ Generated {len(summaries)} summaries")

## 5️⃣ Index Summaries and Details

In [ ]:
# Create indexes
summary_index.create_index(embedding_dim=embedder.get_embedding_dimension(), force_recreate=True)
detail_index.create_index(embedding_dim=embedder.get_embedding_dimension(), force_recreate=True)

# Index summaries (document-level)
print("Indexing summaries...")
summary_docs = []
for item in summaries:
    emb = embedder.embed_text(f"{item['title']} {item['summary']}")
    summary_docs.append({
        'text': item['summary'],
        'embedding': emb,
        'metadata': {
            'doc_id': item['doc_id'],
            'title': item['title'],
            'type': 'summary'
        }
    })

summary_index.index_documents(summary_docs)
print(f"✓ Indexed {len(summary_docs)} summaries")

# Index details (chunk-level)
print("\nIndexing detail chunks...")
detail_docs = []
for item in summaries:
    # Split content into sentences
    sentences = item['full_content'].split('. ')
    for idx, sent in enumerate(sentences):
        if sent.strip():
            emb = embedder.embed_text(sent)
            detail_docs.append({
                'text': sent,
                'embedding': emb,
                'metadata': {
                    'doc_id': item['doc_id'],
                    'title': item['title'],
                    'chunk_idx': idx,
                    'type': 'detail'
                }
            })

detail_index.index_documents(detail_docs)
print(f"✓ Indexed {len(detail_docs)} detail chunks")

print(f"\n✓ Two-stage index complete")
print(f"  Summaries: {len(summary_docs)} documents")
print(f"  Details: {len(detail_docs)} chunks")

## 6️⃣ Document Summary RAG Pipeline

In [ ]:
def document_summary_rag(query: str, summary_top_k: int = 3, detail_top_k: int = 5) -> Dict:
    """
    Two-stage retrieval: summaries → details
    """
    start_time = time.time()
    
    print(f"Query: {query}\n")
    print("=" * 70)
    
    # Stage 1: Search summaries
    print("\nStage 1: Search Document Summaries")
    query_emb = embedder.embed_text(query)
    summary_results = summary_index.vector_search(query_emb, top_k=summary_top_k)
    
    print(f"  Found {len(summary_results)} relevant documents:")
    relevant_doc_ids = []
    for i, result in enumerate(summary_results, 1):
        doc_id = result['metadata']['doc_id']
        relevant_doc_ids.append(doc_id)
        print(f"    {i}. {result['metadata']['title']}")
        print(f"       {result['text'][:80]}...")
    
    # Stage 2: Get details from relevant documents
    print(f"\nStage 2: Retrieve Details from {len(relevant_doc_ids)} Documents")
    
    all_details = []
    for doc_id in relevant_doc_ids:
        # Get detail chunks for this document
        detail_results = detail_index.vector_search(
            query_emb,
            top_k=detail_top_k,
            filter_dict={'doc_id': doc_id}
        )
        all_details.extend([r['text'] for r in detail_results])
        print(f"  ✓ {doc_id}: {len(detail_results)} detail chunks")
    
    print(f"  Total details: {len(all_details)} chunks")
    
    # Generate with both summaries and details
    print("\nStage 3: Generate Answer")
    context = [
        f"Document Summary: {r['text']}" for r in summary_results
    ] + [
        f"Detail: {detail}" for detail in all_details
    ]
    
    answer = llm.generate_with_context(query, context)
    
    total_time = time.time() - start_time
    
    print(f"\n{'=' * 70}")
    print(f"Completed in {total_time:.2f}s")
    print(f"{'=' * 70}")
    
    return {
        'answer': answer,
        'relevant_documents': [r['metadata']['title'] for r in summary_results],
        'summaries': [r['text'] for r in summary_results],
        'details': all_details,
        'metadata': {
            'total_time': total_time,
            'num_docs': len(summary_results),
            'num_details': len(all_details)
        }
    }

print("✓ Document Summary RAG pipeline ready")

## 7️⃣ Test Two-Stage Retrieval

In [ ]:
test_query = "What are the pricing options for AWS AI services?"

result = document_summary_rag(test_query, summary_top_k=SUMMARY_TOP_K, detail_top_k=DETAIL_TOP_K)

print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

print(f"\nRelevant Documents ({len(result['relevant_documents'])}):")
for i, doc in enumerate(result['relevant_documents'], 1):
    print(f"  {i}. {doc}")

print(f"\nSummaries Retrieved:")
for i, summary in enumerate(result['summaries'], 1):
    print(f"  {i}. {summary[:80]}...")

print(f"\nDetails Retrieved: {len(result['details'])} chunks")
for i, detail in enumerate(result['details'][:3], 1):
    print(f"  {i}. {detail[:80]}...")

print(f"\nAnswer:\n{result['answer'][:400]}...")

## 8️⃣ Comparison vs Single-Stage RAG

In [ ]:
# Compare with single-stage (details only)
print("=" * 70)
print("COMPARISON: Two-Stage vs Single-Stage")
print("=" * 70)

# Single-stage: search all details directly
print("\nSingle-Stage RAG:")
single_start = time.time()
query_emb = embedder.embed_text(test_query)
single_results = detail_index.vector_search(query_emb, top_k=10)
single_answer = llm.generate_with_context(test_query, [r['text'] for r in single_results])
single_time = time.time() - single_start

print(f"  Retrieved: {len(single_results)} chunks")
print(f"  Time: {single_time:.2f}s")

# Two-stage: from previous result
print(f"\nTwo-Stage RAG (Summary → Detail):")
print(f"  Stage 1: {result['metadata']['num_docs']} documents")
print(f"  Stage 2: {result['metadata']['num_details']} chunks")
print(f"  Time: {result['metadata']['total_time']:.2f}s")

print(f"\nAdvantages of Two-Stage:")
print(f"  ✓ Document-level filtering first")
print(f"  ✓ More focused detail retrieval")
print(f"  ✓ Better context organization")
print(f"  ✓ Summaries provide overview")

## 9️⃣ Cost Analysis

In [ ]:
evaluator = RAGEvaluator()

print("=" * 70)
print("COST ANALYSIS")
print("=" * 70)

# Summary generation (one-time)
summary_gen_cost = len(summaries) * 0.01  # ~$0.01 per summary

# Query cost
query_cost = evaluator.estimate_cost(
    num_queries=1,
    num_docs=0,
    avg_doc_length=500,
    model_name='sonnet'
)

print(f"\nOne-time costs:")
print(f"  Summary generation: ${summary_gen_cost:.3f}")

print(f"\nPer-query costs:")
print(f"  Embeddings (1 query): $0.00002")
print(f"  Retrieval (2 stages): Included in OpenSearch")
print(f"  Generation: ${query_cost['total_cost']:.4f}")
print(f"  Total: ~${query_cost['total_cost'] + 0.00002:.4f}")

print(f"\nBreak-even: {summary_gen_cost / query_cost['total_cost']:.0f} queries")
print(f"  After this, two-stage is more efficient")

## 🔟 Summary

### Document Summary RAG Pattern

**Two-Stage Approach:**
1. **Stage 1**: Search document summaries (fast filtering)
2. **Stage 2**: Retrieve details from relevant documents

**Benefits:**
- ✅ Efficient document-level filtering
- ✅ Focused detail retrieval
- ✅ Better context organization
- ✅ Lower search cost at scale

**When to Use:**
- Large document collections
- Clear document boundaries
- Need overview + details
- Cost optimization important

**Cost:** ~$0.08-0.12 per query (after initial summary generation)

---

**Pattern #24/37 Complete** ✅